In [3]:
import requests

In [ ]:
query_knowledge_graph("A 67-year-old German woman, who has lived as an expatriate in Cameroon for the past 4 years, presents with palpitations at a tropical medicine clinic in Germany. She also reports transient subcutaneous swellings for the past year. There is no history of fever or constitutional symptoms, and the past medical history is otherwise unremarkable. A 24-hour ECG recording was done elsewhere, which showed paroxysmal supraventricular extrasystoles. Echocardiography revealed a minimal pericardial effusion. On examination, vital signs are normal, and the patient is afebrile. There are no visible subcutaneous swellings. Heart sounds are clear and regular, with no murmurs. No pathological findings are noted on physical examination. Laboratory results show a WBC count of 6.20 x10^9/L, hemoglobin of 13.8 g/dL, platelets of 155 x10^9/L, neutrophils of 45%, eosinophils of 14%, absolute eosinophil count of 0.868 x10^9/L, band neutrophils of 1%, lymphocytes of 36%, monocytes of 4%, and basophils of 0%. The chest radiograph does not show any abnormalities (Table 28.1, Fig 28.1).")

In [14]:
import re
from typing import Any, Dict, List, Tuple

SEP = "<SEP>"

def _normalize_text(s: str) -> str:
    s = (s or "").replace("\\7", "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _split_sep(s: str) -> List[str]:
    s = _normalize_text(s)
    if not s:
        return []
    parts = [p.strip() for p in s.split(SEP)]
    return [p for p in parts if p]

def _keywords_from_metadata(meta: Dict[str, Any]) -> Tuple[List[str], List[str]]:
    kw = (meta or {}).get("keywords", {}) or {}
    hi = [_normalize_text(x).lower() for x in kw.get("high_level", []) if x]
    lo = [_normalize_text(x).lower() for x in kw.get("low_level", []) if x]
    return hi, lo

def _score_text(text: str, hi_kw: List[str], lo_kw: List[str]) -> float:
    """
    Simple, robust scoring:
    - +3 for each high-level keyword hit
    - +1 for each low-level keyword hit
    - small bonus for being more 'sentence-like' (length)
    """
    t = _normalize_text(text).lower()
    if not t:
        return 0.0

    score = 0.0
    for k in hi_kw:
        if k and k in t:
            score += 3.0
    for k in lo_kw:
        if k and k in t:
            score += 1.0

    # mild length bonus so we don't pick ultra-short fragments
    score += min(len(t) / 200.0, 0.5)
    return score

def extract_contexts(result: Dict[str, Any], top_n: int = 5) -> List[str]:
    data = result.get("data", {}) or {}
    meta = result.get("metadata", {}) or {}
    hi_kw, lo_kw = _keywords_from_metadata(meta)

    candidates: List[Tuple[float, str]] = []

    # 1) Relationships first: often already "contextual"
    for rel in data.get("relationships", []) or []:
        src = _normalize_text(rel.get("src_id", ""))
        tgt = _normalize_text(rel.get("tgt_id", ""))
        for desc in _split_sep(rel.get("description", "")):
            # Make the sentence explicit + human-readable
            if src and tgt:
                text = f"{src} → {tgt}: {desc}"
            else:
                text = desc
            candidates.append((_score_text(text, hi_kw, lo_kw), text))

    # 2) Entities: convert entity + description into a clean fact
    for ent in data.get("entities", []) or []:
        name = _normalize_text(ent.get("entity_name", ""))
        etype = _normalize_text(ent.get("entity_type", ""))
        for desc in _split_sep(ent.get("description", "")):
            if name:
                # Keep type if it's informative; avoid cluttering with UNKNOWN
                if etype and etype.upper() != "UNKNOWN":
                    text = f"{name} ({etype}): {desc}"
                else:
                    text = f"{name}: {desc}"
            else:
                text = desc
            candidates.append((_score_text(text, hi_kw, lo_kw), text))

    # 3) Deduplicate (preserve best-scoring version)
    best_by_text: Dict[str, float] = {}
    for score, text in candidates:
        if not text:
            continue
        if text not in best_by_text or score > best_by_text[text]:
            best_by_text[text] = score

    ranked = sorted(best_by_text.items(), key=lambda x: x[1], reverse=True)

    # 4) Pick top N, but skip very low-signal lines
    contexts: List[str] = []
    for text, score in ranked:
        if score < 0.2:
            continue
        contexts.append(text)
        if len(contexts) >= top_n:
            break

    return contexts

def query_knowledge_graph(query: str) -> None:
    url = "http://localhost:9621/query/data"

    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
    }

    payload = {
        "query": query,
        "mode": "mix",
        "only_need_context": True,
        "only_need_prompt": True,
        "response_type": "string",
        "top_k": 20,
        "chunk_top_k": 10,
        "max_entity_tokens": 3500,
        "max_relation_tokens": 2500,
        "max_total_tokens": 6000,
        "enable_rerank": True,
        "include_references": True,
        "include_chunk_content": False,
        "stream": False,
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,  # IMPORTANT: use json= not data=
    )

    response.raise_for_status()
    result = response.json()

    final_result = extract_contexts(result)
    return final_result

In [15]:
query = "A 4-year-old boy from Laos is presented with a rapidly progressing lesion on his cheek. Three days prior to presentation, the family noticed a dark sore on the child’s cheek. The child’s breath smells bad, he is not eating, and he appears listless. The lesion has progressed quickly, eating through the child’s cheek. The child is unimmunized and had a fever and rash about 2 months ago from which he recovered. The family is very poor. On examination, the child appears small and quiet, stunted, and thin. He has a gangrenous lesion that has destroyed part of his upper and lower lips and cheek, exposing his teeth (Fig. 5.1)."
query_knowledge_graph(query)

['Children (person): Children are a specific population that can be infected through contaminated soil. They are susceptible to rotavirus infection and higher filtration capacity than adults. The research studies children as a specific population. Treatment for Crimean-Congo haemorrhagic fever is considered for children. Children are more commonly affected by SSPE than adults. Ivermectin is not prescribed to children under 15 kg.',
 'Infants (population): Infants and small children may exhibit more widespread rash.',
 'Figure 26.4 → School-aged child: Figure 26.4 illustrates a school-aged child with rubella, exanthema, and cervical lymphadenitis.',
 'Children → Schistosomiasis: Children are particularly vulnerable to schistosomiasis and its effects on development.',
 'OHL → Problems: Problems arise, but are sometimes overcome by verrucous lesions which lead to dysphagia.']